In [1]:
import pandas as pd

# Load monthly value-weighted industry returns
df = pd.read_csv(
    '49_Industry_Portfolios.csv',
    skiprows=11,
    engine='python',
    on_bad_lines='skip'
)

# Rename and clean column names
df = df.rename(columns={df.columns[0]: 'Date'})
df.columns = df.columns.str.strip()

# Keep only the five industries used in the assignment
keep_cols = ['Date', 'Oil', 'Banks', 'Txtls', 'Toys', 'Rtail']
df = df[keep_cols]

# Keep only the Value Weighted Monthly Returns
df = df.iloc[:1200].reset_index(drop=True)

print(df.head())
print(df.tail())

     Date      Oil    Banks    Txtls     Toys    Rtail
0  192607    -1.35     4.61     0.50     8.65     0.08
1  192608     3.64    10.71     7.83    16.81    -0.76
2  192609    -3.59    -1.80     2.30     8.33     0.25
3  192610    -1.04   -11.93     0.99    -1.40    -2.19
4  192611     0.05    -2.02     3.19     0.00     6.54
        Date      Oil    Banks    Txtls     Toys    Rtail
1195  202602     8.84    -4.29     4.47     1.11    -3.60
1196  202603    13.09    -2.53   -19.29    -9.25    -2.95
1197  202604    -4.16     8.40     8.62     5.41    13.75
1198  202605    -6.16    -2.28     4.18    -1.35    -2.67
1199  202606    -5.69     8.33    14.77     2.70    -5.59


In [2]:
# Clean Fama-French factor data
factors = pd.read_csv(
    'F-F_Research_Data_5_Factors_2x3.csv',
    skiprows=4,
    engine='python',
    on_bad_lines='skip'
)

# Rename first column to Date and clean column names
factors = factors.rename(columns={factors.columns[0]: 'Date'})
factors.columns = factors.columns.str.strip()

# Keep only monthly observations
factors['Date'] = pd.to_numeric(factors['Date'], errors='coerce')
factors = factors[factors['Date'].between(100000, 999999)]

# Keep only what we need for Question 5
factors = factors[['Date', 'Mkt-RF', 'RF']].reset_index(drop=True)

print(factors.head())
print(factors.tail())

       Date    Mkt-RF        RF
0  196307.0     -0.39      0.27
1  196308.0      5.08      0.25
2  196309.0     -1.57      0.27
3  196310.0      2.54      0.29
4  196311.0     -0.86      0.27
         Date    Mkt-RF        RF
751  202602.0     -1.18      0.28
752  202603.0     -5.18      0.29
753  202604.0      9.95      0.29
754  202605.0      4.91      0.31
755  202606.0     -1.07      0.29


In [3]:
# Make Date the same type in both datasets
df['Date'] = pd.to_numeric(df['Date'], errors='coerce').astype('Int64')
factors['Date'] = pd.to_numeric(factors['Date'], errors='coerce').astype('Int64')

# Merge industry returns with market factor and risk-free rate
data = pd.merge(df, factors, on='Date', how='inner')

print(data.head())
print(data.tail())
print(data.shape)

     Date      Oil    Banks    Txtls     Toys    Rtail    Mkt-RF        RF
0  196307     2.34    -1.58     3.52    -4.89    -1.08     -0.39      0.27
1  196308     3.85     4.15     4.30     4.81     6.65      5.08      0.25
2  196309    -3.62    -3.13    -1.13    -5.30     1.11     -1.57      0.27
3  196310    -0.52    -1.29     5.74    15.38     0.45      2.54      0.29
4  196311    -1.18    -3.31     2.02    -1.17    -1.08     -0.86      0.27
       Date      Oil    Banks    Txtls     Toys    Rtail    Mkt-RF        RF
751  202602     8.84    -4.29     4.47     1.11    -3.60     -1.18      0.28
752  202603    13.09    -2.53   -19.29    -9.25    -2.95     -5.18      0.29
753  202604    -4.16     8.40     8.62     5.41    13.75      9.95      0.29
754  202605    -6.16    -2.28     4.18    -1.35    -2.67      4.91      0.31
755  202606    -5.69     8.33    14.77     2.70    -5.59     -1.07      0.29
(756, 8)


In [4]:
# Columns with returns
return_cols = ['Oil', 'Banks', 'Txtls', 'Toys', 'Rtail', 'Mkt-RF', 'RF']

# Make sure all return columns are numeric
data[return_cols] = data[return_cols].apply(pd.to_numeric, errors='coerce')

# Remove rows with missing values, if any
data = data.dropna(subset=return_cols).reset_index(drop=True)

# Convert percentages to decimals
data[return_cols] = data[return_cols] / 100

# Construct excess returns
industries = ['Oil', 'Banks', 'Txtls', 'Toys', 'Rtail']

for industry_name in industries:
    data[industry_name + '_excess'] = data[industry_name] - data['RF']

data.head()

,Date,Oil,Banks,Txtls,Toys,Rtail,Mkt-RF,RF,Oil_excess,Banks_excess,Txtls_excess,Toys_excess,Rtail_excess
0,196307,0.0234,-0.0158,0.0352,-0.0489,-0.0108,-0.0039,0.0027,0.0207,-0.0185,0.0325,-0.0516,-0.0135
1,196308,0.0385,0.0415,0.0430,0.0481,0.0665,0.0508,0.0025,0.0360,0.0390,0.0405,0.0456,0.0640
2,196309,-0.0362,-0.0313,-0.0113,-0.0530,0.0111,-0.0157,0.0027,-0.0389,-0.0340,-0.0140,-0.0557,0.0084
3,196310,-0.0052,-0.0129,0.0574,0.1538,0.0045,0.0254,0.0029,-0.0081,-0.0158,0.0545,0.1509,0.0016
4,196311,-0.0118,-0.0331,0.0202,-0.0117,-0.0108,-0.0086,0.0027,-0.0145,-0.0358,0.0175,-0.0144,-0.0135


In [5]:
# Matrix with the 5 industry excess returns
Re = data[
    ['Oil_excess', 'Banks_excess', 'Txtls_excess', 'Toys_excess', 'Rtail_excess']
].to_numpy()

# Market excess return
f = data['Mkt-RF'].to_numpy()

print(Re.shape)
print(f.shape)

(756, 5)
(756,)


In [6]:
# Number of observations and assets
T = len(f)
N = Re.shape[1]

print("Observations:", T)
print("Industry portfolios:", N)

# Starting values for GMM parameters
mu_start = np.mean(f)
c_start = 1.0

theta_start = np.array([mu_start, c_start])

print("Starting mu:", mu_start)
print("Starting c:", c_start)

Observations: 756
Industry portfolios: 5


NameError: name 'np' is not defined

In [ ]:
def moments(theta):
    mu, c = theta

    # CAPM stochastic discount factor
    m = 1 - c * (f - mu)

    # Moment 1: mean of market excess return
    moment_market = f - mu

    # Moments 2-6: pricing errors for the 5 industries
    moment_assets = m[:, None] * Re

    # Put all 6 moment conditions together
    g = np.column_stack((moment_market, moment_assets))

    return g

In [ ]:
g_start = moments(theta_start)

print(g_start.shape)
print(g_start.mean(axis=0))

(756, 6)
[-1.61540387e-18  5.00221924e-03  3.94777694e-03  3.39603943e-03
  1.86505028e-03  5.36867241e-03]


In [ ]:
from scipy.optimize import minimize

In [ ]:
# Step 1 weighting matrix: identity matrix
W1 = np.eye(6)

# GMM objective function
def gmm_objective(theta, W):
    g = moments(theta)
    g_bar = g.mean(axis=0)

    return g_bar @ W @ g_bar

In [ ]:
# Estimate GMM Step 1
result1 = minimize(
    gmm_objective,
    theta_start,
    args=(W1,),
    method='BFGS'
)

theta1 = result1.x
mu1, c1 = theta1

print("Success:", result1.success)
print("Step 1 mu:", mu1)
print("Step 1 c:", c1)
print("Objective value:", result1.fun)

Success: True
Step 1 mu: 0.005970281253338007
Step 1 c: 2.78904875969146
Objective value: 1.3445530799500513e-05


In [ ]:
# Moment conditions evaluated at Step 1 estimates
g1 = moments(theta1)

# Newey-West estimator of S
def newey_west_S(g, q):
    T = g.shape[0]

    # Gamma_0
    S = (g.T @ g) / T

    # Autocovariances
    for j in range(1, q + 1):
        weight = 1 - j / (q + 1)

        Gamma_j = (g[j:].T @ g[:-j]) / T

        S += weight * (Gamma_j + Gamma_j.T)

    return S

# Automatic Newey-West lag choice
q = int(np.floor(4 * (T / 100) ** (2 / 9)))

S1 = newey_west_S(g1, q)

# Efficient Step 2 weighting matrix
W2 = np.linalg.inv(S1)

print("Newey-West lags:", q)
print("S shape:", S1.shape)
print("W2 shape:", W2.shape)

Newey-West lags: 6
S shape: (6, 6)
W2 shape: (6, 6)


In [ ]:
# Estimate efficient GMM Step 2
result2 = minimize(
    gmm_objective,
    theta1,
    args=(W2,),
    method='BFGS'
)

theta2 = result2.x
mu2, c2 = theta2

print("Success:", result2.success)
print("Step 2 mu:", mu2)
print("Step 2 c:", c2)
print("Objective value:", result2.fun)

Success: True
Step 2 mu: 0.006370143850385061
Step 2 c: 3.5705439526956346
Objective value: 0.006319145590697498


In [ ]:
# Average moment conditions
def gbar(theta):
    return moments(theta).mean(axis=0)

# Numerical Jacobian
def numerical_jacobian(func, theta, h=1e-6):
    theta = np.asarray(theta, dtype=float)
    D = np.zeros((6, 2))

    for j in range(2):
        step = np.zeros(2)
        step[j] = h

        D[:, j] = (
            func(theta + step) - func(theta - step)
        ) / (2 * h)

    return D


# ---------- Step 1 standard errors ----------
D1 = numerical_jacobian(gbar, theta1)

A1 = D1.T @ W1 @ D1
B1 = D1.T @ W1 @ S1 @ W1 @ D1

V1 = np.linalg.inv(A1) @ B1 @ np.linalg.inv(A1) / T
se1 = np.sqrt(np.diag(V1))


# ---------- Step 2 standard errors ----------
g2 = moments(theta2)
S2 = newey_west_S(g2, q)

D2 = numerical_jacobian(gbar, theta2)

A2 = D2.T @ W2 @ D2
B2 = D2.T @ W2 @ S2 @ W2 @ D2

V2 = np.linalg.inv(A2) @ B2 @ np.linalg.inv(A2) / T
se2 = np.sqrt(np.diag(V2))


print("Step 1:")
print("mu =", mu1, "SE =", se1[0])
print("c  =", c1,  "SE =", se1[1])

print("\nStep 2:")
print("mu =", mu2, "SE =", se2[0])
print("c  =", c2,  "SE =", se2[1])

Step 1:
mu = 0.005970281253338007 SE = 0.0016631091330655144
c  = 2.78904875969146 SE = 1.0760570176663236

Step 2:
mu = 0.006370143850385061 SE = 0.0016445565498244908
c  = 3.5705439526956346 SE = 1.0276678431862944


In [ ]:
from scipy.stats import chi2

# Average moment conditions at Step 2 estimates
g_bar2 = moments(theta2).mean(axis=0)

# Hansen J-statistic
J_stat = T * (g_bar2 @ W2 @ g_bar2)

# Degrees of freedom: moments - parameters
J_df = 6 - 2

# P-value
J_pvalue = chi2.sf(J_stat, J_df)

print("Hansen J-statistic:", J_stat)
print("Degrees of freedom:", J_df)
print("P-value:", J_pvalue)

Hansen J-statistic: 4.777274066567308
Degrees of freedom: 4
P-value: 0.3109232294820088


In [ ]:
# CAPM slope restriction: c = mu / Var(f)

var_f = np.var(f, ddof=0)

# Difference between estimated c and restricted CAPM value
restriction = c2 - mu2 / var_f

# Delta-method gradient with respect to (mu, c)
gradient = np.array([
    -1 / var_f,
    1
])

# Standard error of the restriction
var_restriction = gradient @ V2 @ gradient
se_restriction = np.sqrt(var_restriction)

# Wald z-statistic
z_stat = restriction / se_restriction

# Two-sided p-value
p_value_slope = chi2.sf(z_stat**2, df=1)

print("Estimated c:", c2)
print("CAPM restricted c:", mu2 / var_f)
print("Difference:", restriction)
print("SE difference:", se_restriction)
print("z-statistic:", z_stat)
print("p-value:", p_value_slope)

Estimated c: 3.5705439526956346
CAPM restricted c: 3.2022858867789283
Difference: 0.3682580659167063
SE difference: 0.4938950534451193
z-statistic: 0.7456200732280192
p-value: 0.45589695356000637
